# 06 — Live pitch latency

Size the window/hop tradeoff for the browser AudioWorklet. Latency is the priority metric for the practice loop.

In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "tests").exists():
    ROOT = ROOT.parent
AUDIO_DIR = ROOT / "tests" / "fixtures" / "audio"
GOLDEN_DIR = ROOT / "tests" / "fixtures" / "golden"
print("project root:", ROOT)


project root: /home/anton/Projects/music-teacher


In [2]:
import time, statistics
from music_teacher.evaluation.fixtures import Fixture, FixtureNote, synthesize
from music_teacher.transcription.monophonic import track_pitch, yin_pitch

audio = synthesize(Fixture(name="bench", notes=[FixtureNote(start=0.0, end=4.0, pitch=69)], timbre="flute", trailing_silence=0.0))
print(audio.shape)


(88200,)


In [3]:
for frame_length, hop in ((1024, 128), (2048, 256), (4096, 256)):
    start = time.perf_counter()
    track_pitch(audio, frame_length=frame_length, hop_length=hop)
    elapsed = time.perf_counter() - start
    n_frames = len(range(0, audio.shape[0] - frame_length + 1, hop))
    algorithmic_ms = (frame_length + hop) / 22050 * 1000
    print(f"window={frame_length:5d} hop={hop:4d}  alg.latency={algorithmic_ms:6.1f} ms  proc/frame={elapsed / n_frames * 1000:.3f} ms")


window= 1024 hop= 128  alg.latency=  52.2 ms  proc/frame=0.062 ms
window= 2048 hop= 256  alg.latency= 104.5 ms  proc/frame=0.092 ms
window= 4096 hop= 256  alg.latency= 197.4 ms  proc/frame=0.157 ms


In [4]:
# simulate streaming: run YIN frame by frame and collect f0
from collections import deque

frame_length, hop = 2048, 256
f0s = []
for start in range(0, audio.shape[0] - frame_length, hop):
    frequency, confidence = yin_pitch(audio[start:start + frame_length])
    f0s.append((start / 22050, frequency, confidence))

voiced = [(t, f) for t, f, c in f0s if f > 0 and c > 0.5]
print(f"{len(voiced)}/{len(f0s)} voiced frames; median f0 = {statistics.median(f for _, f in voiced):.1f} Hz")
print(f"algorithmic latency = {(frame_length + hop) / 22050 * 1000:.1f} ms")


337/337 voiced frames; median f0 = 440.1 Hz
algorithmic latency = 104.5 ms


Browser target: keep the analysis window at 2048/256 at 44.1 kHz and resample-free worklet buffers; YIN itself is far below the latency budget.